# KAT-VAD v2 — P6 / **pilot batch 2: A2 + A3** (seed 2099, encoder V2-S)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§4** (pilot read-out), **§13 D16**, **§14 O1–O7**, **§15
Amendment 6 (G4–G5)**. Needs: batch 1 finished (`p6_pilot_batch1.ipynb`: v2 dataset dir, KNN cache, A0/A1 checkpoints
and `diag.json`) and the full-T2 V2-S cache (`p4_s_full_t2_d6.ipynb` step 2).

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps | — |
| 1 | preflight; stage the v2 dir, KNN, CLIP + V2-S caches to VM disk | no |
| 2 | **bake A2** (`fit --crn none --motion S`) and **A3** (`--crn R2 --motion S`); `apply --stride 3` on `dota_cap_dev` | no |
| 3 | **train A2 and A3**, phase 4's recipe, seed 2099, chunked + resumable, verified on Drive (J10) | **yes** |
| 4 | **diagnostics** O1–O5: A2/A3, and A0/A1 re-read, all with O5 on `dota_cap_dev` (G5) | yes (or slow CPU) |
| 5 | record → stop | — |

**DoTA is not printed (D4).** DoTA-CAP-dev is read unlabelled, for the source-shortcut probe only (O5).
`ρ_u ≈ 0` is recorded, never fixed by `lr` or `c` (§4). Everything new goes to `Thesis-V2/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE), v1 outputs
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), data/, cache/, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V2_D6_SHUFFLE_SEED'), 'this upload predates Amendment 6 -- re-upload branch v2'

DATASET    = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL     # 3
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT          # 5
NUM_EPOCHS, CHUNK_EPOCHS = 20, 5                         # phase 4's recipe (O7)
SEED       = 2099                                        # D4
CRN        = 'R2'                                        # D11
ENC        = constants.VIDEOMAE_ENCODER_S                # E2(d) pick (§15 G1)
MOTION_DIM = constants.VIDEOMAE_ARCH[ENC][0]             # 384

CLIP_DIR  = constants.CLIP_CACHE_DIR / DATASET
S_T2      = constants.VIDEO_CACHE_DIR / ENC / f'{DATASET}_s{constants.FRAME_STRIDE}_squash'
S_CAP     = constants.VIDEO_CACHE_DIR / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash'
DOTA      = constants.DATA_ROOT / 'DoTA'
LABELS    = DOTA / 'labels_s8'                            # defs.json only: no DoTA label is read
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_DATA   = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'     # batch 1 published these three
V2_CACHE  = Path(DRIVE_V2) / 'cache'
KNN_V2    = V2_CACHE / 'knn' / f'{DATASET}_v2' / constants.KNN_CACHE_FILENAME
A1_T2     = V2_CACHE / 'v2' / f'A1_{CRN}' / DATASET
A1_DOTA   = V2_CACHE / 'v2' / f'A1_{CRN}' / 'DoTA_s3_from_s1'
TAG       = {'A2': f'A2_S', 'A3': f'A3_{CRN}_S'}
BAKED     = {arm: (V2_CACHE / 'v2' / TAG[arm] / DATASET, V2_CACHE / 'v2' / TAG[arm] / 'DoTA_CAP_s3_from_s1')
             for arm in TAG}
PILOT     = constants.OUTPUT_ROOT / 'v2_pilot'           # <arm>/s2099/{run files, diag/, diag_cap/}
LOCAL = Path('/content/pilot2')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('CLIP_DIR', 'S_T2', 'S_CAP', 'V2_DATA', 'KNN_V2', 'PILOT'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training will be unusably slow'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2)."""
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive and fsync every file (C10)."""
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight + stage

In [ ]:
from core.data.v2_splits import load_split
from core.tools import build_v2_inputs, e2d_probe, v2_dataset, v2_diagnostics

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 15. Amendment 6' in addendum, 'upload predates §15'
cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)
assert len(cap_dev) == 569, len(cap_dev)
pick = json.loads((constants.OUTPUT_ROOT / 'REPORTS' / 'v2_E2d' / e2d_probe.READOUT_JSON).read_text())
assert pick['decision']['pick'] == ENC, pick['decision']
for arm in ('A0', 'A1'):                                  # batch 1 must be finished and verified (J10)
    d = PILOT / arm / f's{SEED}'
    last = metrics_last(d / 'metrics.jsonl')
    assert last and last['epoch'] + 1 == NUM_EPOCHS and ckpt_step(d / 'checkpoint_last.pt') == last['global_step'], \
        f'batch 1 {arm} is not finished -- run p6_pilot_batch1.ipynb first'
    assert (d / 'diag' / v2_diagnostics.DIAG_JSON).exists(), f'batch 1 {arm} has no diag.json'
for path in (V2_DATA / v2_dataset.MANIFEST_FILENAME, KNN_V2, A1_T2, A1_DOTA, LABELS / constants.DEFS_FILENAME):
    assert path.exists(), f'missing {path} (published by batch 1)'

L_V2 = stage(V2_DATA, LOCAL / 'stage' / 'corpus_v2')
L_KNN = stage(KNN_V2, LOCAL / 'stage' / 'knn_v2.npz')
sources = build_v2_inputs.t2_all_sources(L_V2)
assert all((S_T2 / f'{v}.npy').exists() for v in sources), 'full-T2 V2-S cache incomplete -- p4_s_full_t2_d6 step 2'
L_CLIP = stage(CLIP_DIR, LOCAL / 'stage' / 'clip')
L_S_T2 = stage(S_T2, LOCAL / 'stage' / 's_t2', sources)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_cap', cap_dev)
L_S_CAP = stage(S_CAP, LOCAL / 'stage' / 's_cap', cap_dev)
L_A1_T2 = stage(A1_T2, LOCAL / 'stage' / 'A1_T2')
L_A1_DOTA = stage(A1_DOTA, LOCAL / 'stage' / 'A1_DOTA')
print(f'T2 sources {len(sources)} | dota_cap_dev {len(cap_dev)} (unlabelled use only)')

## 2. Bake A2 and A3 (G4)

`fit` on the **same** v2 dataset dir as A1 (statistics on T2-train minus T2-val, `train_ids_sha1` recorded); every T2
source is baked. `apply --stride 3` bakes `dota_cap_dev` from the stride-1 DoTA-CAP V2-S cache (O5 only here).

In [ ]:
CRN_OF = {'A2': 'none', 'A3': CRN}
L_BAKED = {arm: (LOCAL / 'stage' / f'{arm}_T2', LOCAL / 'stage' / f'{arm}_CAP') for arm in CRN_OF}
ids_file = LOCAL / 'dota_cap_dev.txt'
ids_file.write_text(''.join(f'{v}\n' for v in cap_dev))
a1 = json.loads((L_A1_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
for arm, crn in CRN_OF.items():
    t2_out, cap_out = L_BAKED[arm]
    if not (t2_out / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'fit', '--t2-dir', L_V2,
             '--clip-dir', L_CLIP, '--video-dir', L_S_T2, '--crn', crn, '--motion', ENC, '--out-dir', t2_out])
    if not (cap_out / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', t2_out,
             '--clip-dir', L_DOTA_S1, '--video-dir', L_S_CAP, '--stride', constants.V2_E1_STRIDE_BC,
             '--ids-file', ids_file, '--out-dir', cap_out])
    m = json.loads((t2_out / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    print(f"{arm}: crn {m['crn']} motion {m['motion']} width {m['width']} s={m['s']:.4f} c={m['c']:.4f} on "
          f"{m['train_sources']} T2-train sources (val {m['t2_val_sources']}); baked {m['baked_ids']}")
    assert m['arm'] == arm and m['motion'] == ENC and m['width'] == 512 + MOTION_DIM
    assert m['train_ids_sha1'] == a1['train_ids_sha1'], f'{arm} fitted on another train split than A1'
    publish(t2_out, BAKED[arm][0])
    publish(cap_out, BAKED[arm][1])

## 3. Train A2 and A3 — chunked, resumable, verified

Phase 4's recipe + `v2.motion` + `model.motion_dim`. 5 epochs per call, `--resume` in between, synced to Drive after every
chunk; skip/resume reads the **checkpoint's own `global_step`** (Amendment 3). **Re-run this cell after a disconnect.**

In [ ]:
MOTION = ['--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={MOTION_DIM}']
ARMS = {
    'A2': {'clip': L_BAKED['A2'][0], 'extra': MOTION},
    'A3': {'clip': L_BAKED['A3'][0], 'extra': ['--set', f'v2.crn={CRN}', *MOTION]},
}


def train_cmd(arm, out_dir):
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
            '--set', f'train.num_epochs={NUM_EPOCHS}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={KERNEL}', '--set', f'loss.mil_topk_pct={TOPK_PCT}',
            '--set', 'kip.enabled=false', *ARMS[arm]['extra'],
            '--data-dir', L_V2, '--clip-dir', ARMS[arm]['clip'], '--knn-cache', L_KNN,
            '--output-dir', out_dir]


def sync(local, drive):
    drive.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (local / name).exists():
            publish(local / name, drive / name)


def final_step(local):
    return metrics_last(local / 'metrics.jsonl')['global_step']


for arm in ARMS:
    drive_dir, local = PILOT / arm / f's{SEED}', LOCAL / 'runs' / arm
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if last and last['epoch'] + 1 == NUM_EPOCHS and ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        print(f'{arm}: finished + verified on Drive (step {last["global_step"]}) -- skipping')
        continue
    local.mkdir(parents=True, exist_ok=True)
    if not (local / 'checkpoint_last.pt').exists() and last and \
            ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            shutil.copy2(drive_dir / name, local / name)          # resume from a consistent Drive copy
    while True:
        done = (metrics_last(local / 'metrics.jsonl') or {'epoch': -1})['epoch'] + 1
        if done >= NUM_EPOCHS:
            break
        cmd = train_cmd(arm, local)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'=== {arm}: epochs {done} -> {min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd, log=LOCAL / f'{arm}_train.log', quiet_steps=True)
        sync(local, drive_dir)
    assert ckpt_step(local / 'checkpoint_last.pt') == final_step(local), f'{arm}: J10 fails locally'
    sync(local, drive_dir)
    print(f'{arm}: trained to step {final_step(local)}, synced')

# Config check: A2 == A0 but the motion switches; A3 == A1 but the motion switches.
cfg = {arm: yaml.safe_load((PILOT / arm / f's{SEED}' / 'config.yaml').read_text()) for arm in ('A0', 'A1', 'A2', 'A3')}


def diff(a, b):
    out = set()
    for section in set(a) | set(b):
        sa, sb = a.get(section) or {}, b.get(section) or {}
        out |= {f'{section}.{k}' for k in set(sa) | set(sb) if sa.get(k) != sb.get(k)}
    return out


motion_keys = {'v2.motion', 'model.motion_dim'}
assert diff(cfg['A0'], cfg['A2']) == motion_keys, f'A2 differs from A0 beyond motion: {diff(cfg["A0"], cfg["A2"])}'
assert diff(cfg['A1'], cfg['A3']) == motion_keys, f'A3 differs from A1 beyond motion: {diff(cfg["A1"], cfg["A3"])}'
for arm in ARMS:
    drive_dir = PILOT / arm / f's{SEED}'
    step, last = ckpt_step(drive_dir / 'checkpoint_last.pt'), metrics_last(drive_dir / 'metrics.jsonl')
    assert step == last['global_step'] and last['epoch'] + 1 == NUM_EPOCHS, f'{arm}: J10 FAILS on Drive'
    print(f'{arm}: Drive checkpoint step {step} == metrics last step (J10)')
print('configs: A2 == A0 and A3 == A1, but v2.motion + model.motion_dim')

## 4. Diagnostics (O1–O5), O5 on `dota_cap_dev` for all four arms (G5)

A2/A3 read A0's batch-1 `diag.json` for the O1 margin and O4 comparison. A0 and A1 are **re-read** with
`--dota-split dota_cap_dev` into `diag_cap/` (their T2-val side is unchanged; only O5's DoTA side moves), so the four
arms' source-shortcut numbers are on the same 569 clips. Batch 1's `diag/` is left untouched.

In [ ]:
L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME, LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent
INPUTS = {'A0': (L_CLIP, L_DOTA_S1), 'A1': (L_A1_T2, L_A1_DOTA),
          'A2': L_BAKED['A2'], 'A3': L_BAKED['A3']}
A0_DIAG = PILOT / 'A0' / f's{SEED}' / 'diag' / v2_diagnostics.DIAG_JSON


def diagnose(arm):
    out = PILOT / arm / f's{SEED}' / 'diag_cap'
    if (out / v2_diagnostics.DIAG_JSON).exists():
        print(f'{arm}: reusing {out / v2_diagnostics.DIAG_JSON}')
    else:
        t2_in, dota_in = INPUTS[arm]
        run([sys.executable, '-m', 'core.tools.v2_diagnostics', '--run', f'{arm}_s{SEED}',
             PILOT / arm / f's{SEED}' / 'checkpoint_last.pt',
             '--data-dir', L_V2, '--t2-input-dir', t2_in, '--dota-input-dir', dota_in,
             '--dota-s1-dir', L_DOTA_S1, '--dota-data-dir', L_DOTA_LABELS,
             '--dota-split', constants.V2_SPLIT_DOTA_CAP_DEV,
             *(['--a0-diag', A0_DIAG] if arm != 'A0' else []), '--out-dir', out, '--device', 'auto'],
            log=LOCAL / f'{arm}_diag.log')
    print((out / v2_diagnostics.DIAG_MD).read_text())
    return json.loads((out / v2_diagnostics.DIAG_JSON).read_text())


diags = {arm: diagnose(arm) for arm in ('A0', 'A1', 'A2', 'A3')}
batch1 = {arm: json.loads((PILOT / arm / f's{SEED}' / 'diag' / v2_diagnostics.DIAG_JSON).read_text())
          for arm in ('A0', 'A1')}
for arm, b1 in batch1.items():   # the T2-val side is the same forward pass: it must reproduce batch 1
    for key in ('micro', 'macro'):
        assert abs(b1['guardrails'][key] - diags[arm]['guardrails'][key]) < 1e-4, (arm, key)  # GPU float noise only
print('A0/A1 re-read reproduces batch 1 on T2-val')

In [ ]:
def share(r, key, field):
    v = (r.get('motion_share') or {}).get(key)
    return '—' if v is None else f'{v[field]:.4g}'


rows = []
for arm, r in diags.items():
    g = r['guardrails']
    rows.append(f"| {arm} | {g['micro']:.4f} | {g['macro']:.4f} | {g['clip_oracle_micro']:.4f} | "
                f"{g['window_level_auc']:.4f} | {'PASS' if g['pass'] else 'FAIL'} | "
                f"{r['position_r2']['v_t']:.4f} | {r['source_shortcut_auc']['v_t']:.4f} | "
                f"{r['source_shortcut_auc']['y_bin']:.4f} | {share(r, 'motion_share', 'last')} / "
                f"{share(r, 'motion_share', 'max')} | {share(r, 'w_u_norm', 'last')} |")
table = '\n'.join(['| arm | T2-val micro | macro | clip oracle | window AUC | guardrails | pos R² V^t | '
                   'shortcut V^t (CAP) | shortcut y^bin (CAP) | ρ_u last / max | ‖W_u‖ last |',
                   '|---|---|---|---|---|---|---|---|---|---|---|', *rows])
text = (f'# v2 pilot batch 2 (seed {SEED}, encoder {ENC}) — T2-val only (D4)\n\n{table}\n\n'
        f'O5 for all four arms on `dota_cap_dev` (569 clips, unlabelled; Amendment 6 G5). '
        f'A0/A1 rows = batch 1 checkpoints re-read; their T2-val columns reproduce batch 1.\n')
(PILOT / 'batch2_readout.md').write_text(text)
print(text)

## 5. Record — then **stop**

Paste `v2_pilot/batch2_readout.md` and the four `diag_cap/diag.md` back to Claude. A guardrail FAIL blocks P7 (§5)
until explained; it is not fixed by changing `lr`, `c` or the recipe (§4). `ρ_u ≈ 0` is a result, not a bug to tune.